<img src="https://raw.githubusercontent.com/OmarLi-360/powerbi-tableau-analista/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 1 · Conectar y perfilar
**Analista de datos · Power BI + Tableau** — La calidad de cada columna antes de cargar · ⏱ 25 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/01_conectar_y_perfilar.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/powerbi-tableau-analista/#clave-01)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Ver la calidad de cada columna con el tipo automático y con el corregido.
2. Leer la distribución para encontrar variantes y llaves sucias.
3. Comprobar que el archivo del curso llega limpio.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/powerbi-tableau-analista"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

In [ ]:
# @title Preparar el perfil de columnas (como Power Query)
erp = pd.read_csv(ruta("ventas_erp_sucio.csv"), dtype=str, keep_default_na=False)
def es_entero(x): return x.strip().lstrip("-").isdigit()
def es_fecha(x):
    for f in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            pd.to_datetime(x, format=f); return True
        except ValueError:
            pass
    return False
def detectar(col):
    m = [x for x in col.head(200) if x != ""]
    if m and sum(map(es_entero, m)) / len(m) >= 0.9: return "Número entero"
    if m and sum(map(es_fecha, m)) / len(m) >= 0.9: return "Fecha"
    return "Texto"
def limpiar(nombre, x):
    if x == "": return x
    if nombre in ("Valor Neto", "Cantidad"): return x.replace("$", "").replace(".", "").replace("und", "").strip()
    if nombre == "Fecha" and es_entero(x): return (pd.Timestamp("1899-12-30") + pd.Timedelta(days=int(x))).strftime("%Y-%m-%d")
    if nombre == "Cod Cliente": return x.strip().upper()
    return x
TIPO_OK = {"Linea": "Número entero", "Cantidad": "Número entero", "Valor Neto": "Número entero", "Fecha": "Fecha"}
def perfil(tabla, modo):
    filas = []
    for c in tabla.columns:
        col = tabla[c] if modo == "Automáticos" else tabla[c].map(lambda x: limpiar(c, x))
        tipo = detectar(tabla[c]) if modo == "Automáticos" else TIPO_OK.get(c, "Texto")
        ok = col[col != ""]
        validos = ok.map(es_entero).sum() if tipo == "Número entero" else ok.map(es_fecha).sum() if tipo == "Fecha" else len(ok)
        filas.append({"Columna": c, "Tipo": tipo, "Válidos": int(validos), "Errores": int(len(ok) - validos),
                      "Vacíos": int((col == "").sum()), "Distintos": col[col != ""].nunique()})
    return pd.DataFrame(filas).set_index("Columna")
display(Markdown(f"Exportación leída: **{miles(len(erp))} filas** y {erp.shape[1]} columnas, todo como texto (como llega el CSV)."))

## 1 · Calidad de columna
Es lo que muestra Power Query con **Vista › Calidad de columna**: el porcentaje de valores válidos, con error y vacíos. Elige si quieres ver los tipos que adivina el paso automático **Tipo cambiado** o los tipos después de corregir en la consulta.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué columnas llegan con problemas? |
> | **Qué se grafica** | Barras apiladas al 100 %: válidos, errores y vacíos por columna |
> | **Por qué esta forma** | Es la misma barra que Power Query dibuja sobre cada columna, en un solo gráfico. |
> | **Qué se descartó** | Una tabla de conteos: los porcentajes pequeños de error pasan desapercibidos. |
> | **Cómo se lee** | Cualquier segmento rojo es un valor que no encaja en el tipo. El gris son vacíos: pueden ser válidos (una venta sin valor) o un problema. |
> | **En Excel 365** | Editor de Power Query › Vista › Calidad de columna, con el perfil basado en todo el conjunto de datos. |

In [ ]:
# @title Gráfico 1 · Calidad de columna
tipos = "Automáticos"  # @param ["Automáticos", "Corregidos"]
p = perfil(erp, tipos)
pct = p[["Válidos", "Errores", "Vacíos"]].div(len(erp))
fig = go.Figure()
for col, color in [("Válidos", BIEN), ("Errores", MAL), ("Vacíos", GRIS)]:
    fig.add_bar(name=col, y=[f"{c} ({p.Tipo[c]})" for c in p.index], x=pct[col], orientation="h", marker_color=color,
                text=[f"{v:.1%}".replace(".", ",") if v > 0.005 else "" for v in pct[col]])
con_error = p[p.Errores > 0]
fig.update_layout(barmode="stack", xaxis_tickformat=".0%", height=420, yaxis=dict(autorange="reversed"), legend=dict(orientation="h", y=-0.12),
                  title=(f"Tipos automáticos: {len(con_error)} columnas con errores ({', '.join(con_error.index)})" if len(con_error)
                         else "Tipos corregidos: ninguna columna con errores"))
fig.show()
display(p.style.format({"Válidos": miles, "Errores": miles, "Vacíos": miles, "Distintos": miles}))

## 2 · Distribución: lo que el tipo no muestra

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿De cuántas formas llega escrita la ciudad? |
> | **Qué se grafica** | Barras: filas por cada valor distinto de Ciudad · Magenta: variantes que no son el nombre oficial |
> | **Por qué esta forma** | Con el tipo correcto no hay errores, pero la distribución muestra 16 valores donde debería haber 6. |
> | **Qué se descartó** | Mirar solo la calidad de columna: diría 100 % válido. |
> | **Cómo se lee** | Cada barra magenta es una ciudad «nueva» en el tablero. Se normaliza con Reemplazar valores o con un maestro de sedes. |
> | **En Excel 365** | Vista › Distribución de columnas; y en el perfil, la lista de valores más frecuentes. |

In [ ]:
# @title Gráfico 2 · Distribución de Ciudad
dist = erp["Ciudad"].value_counts().sort_values()
oficiales = set(sedes.ciudad)
fig = go.Figure(go.Bar(x=dist.values, y=[f"«{c}»" for c in dist.index], orientation="h",
                       marker_color=[AZUL if c in oficiales else MAGENTA for c in dist.index],
                       text=dist.values, textposition="outside", cliponaxis=False))
fig.update_layout(title=f"{len(dist)} valores distintos para {len(oficiales)} sedes: {miles(dist[[c not in oficiales for c in dist.index]].sum())} filas con variantes",
                  xaxis_title="Filas", height=480)
dar_aire(fig, dist.max())
fig.show()

## 3 · El archivo del curso

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿ventas.csv está listo para el modelo? |
> | **Qué se grafica** | Barras: valores distintos de cada columna de ventas.csv frente al número de filas |
> | **Por qué esta forma** | Una llave como id_linea debe tener tantos valores distintos como filas; los atributos, pocos. |
> | **Qué se descartó** | Revisar solo los tipos: no muestra si una llave se repite. |
> | **Cómo se lee** | id_linea llega a 17.106 (una por fila): es llave. canal y estado tienen 3: son atributos. |
> | **En Excel 365** | Distribución de columnas: «distintos» y «únicos» bajo cada encabezado. |

In [ ]:
# @title Gráfico 3 · Distintos por columna en ventas.csv
cols = ["id_linea", "id_factura", "fecha", "id_cliente", "sku", "id_vendedor", "id_sede", "canal", "estado"]
dist_v = ventas[cols].nunique()
fig = go.Figure(go.Bar(x=dist_v.index, y=dist_v.values, marker_color=[AZUL if v == len(ventas) else GRIS for v in dist_v.values],
                       text=[miles(v) for v in dist_v.values], textposition="outside"))
fig.add_hline(y=len(ventas), line_dash="dash", line_color=MAGENTA, annotation_text=f"Filas: {miles(len(ventas))}")
fig.update_layout(title="Solo id_linea es única en cada fila: es la llave de la tabla Ventas", yaxis_type="log", yaxis_title="Valores distintos (escala log)", height=380)
fig.show()

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · Con los tipos automáticos, ¿cuántos errores tiene la columna Fecha?
respuesta = "Elige una opción"  # @param ["Elige una opción", "0", "38", "54", "75"]
revisar(respuesta, str(perfil(erp, "Automáticos").loc["Fecha", "Errores"]), """Son las fechas que llegaron como número serial (45299): no encajan en el tipo fecha.""")

In [ ]:
# @title Reto 2 · Valor Neto quedó como texto sin errores. ¿Qué pasa si la cargas así?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Nada, está bien", "No se puede sumar: hay que limpiar «$» y puntos y cambiar a número", "Power BI la convierte sola en el informe", "Se borran los vacíos"]
revisar(respuesta, "No se puede sumar: hay que limpiar «$» y puntos y cambiar a número", """Sin errores no significa listo: el tipo también debe ser el correcto.""")

In [ ]:
# @title Reto 3 · ¿Qué columna de ventas.csv es la llave de la tabla?
respuesta = "Elige una opción"  # @param ["Elige una opción", "id_factura", "id_linea", "id_cliente", "fecha"]
revisar(respuesta, "id_linea", """Es la única con tantos valores distintos como filas.""")

## Apóyate en IA
Para entender un error de Power Query:

```text
En Power Query, la columna [NOMBRE] tiene [X] % de errores y el valor que lo causa es [VALOR].
Explica la causa y los pasos (menú por menú, en español) para corregirlo en la consulta.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Primero se perfila, después se carga.
* Calidad de columna muestra errores y vacíos; la distribución muestra variantes y llaves.
* Se corrige en la consulta, no en el informe.

**Siguiente:** [Clave 2 · Transformar para el modelo](https://omarli-360.github.io/powerbi-tableau-analista/#clave-02) · [Cuaderno 02](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/02_transformar_para_el_modelo.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>